In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
# -----------------------------
# 1. Load predefined dataset
# -----------------------------
num_words = 10000
max_len = 100

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

# Convert labels to sequences (repeat label across time)
y_train_seq = np.repeat(y_train[:, None], max_len, axis=1)
y_test_seq = np.repeat(y_test[:, None], max_len, axis=1)

In [3]:
# -----------------------------
# 2. Encoder
# -----------------------------
encoder_inputs = Input(shape=(max_len,), name="encoder_input")
embedding = Embedding(num_words, 128, mask_zero=True)

encoder_emb = embedding(encoder_inputs)

encoder_lstm = LSTM(64, return_state=True, name="encoder_lstm",use_cudnn=False)
_, state_h, state_c = encoder_lstm(encoder_emb)

encoder_states = [state_h, state_c]

In [4]:
# -----------------------------
# 3. Decoder
# -----------------------------
decoder_inputs = Input(shape=(max_len, 1), name="decoder_input")

decoder_lstm = LSTM(
    64,
    return_sequences=True,
    return_state=True,
    use_cudnn=False,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = Dense(1, activation="sigmoid")
decoder_outputs = decoder_dense(decoder_outputs)

In [5]:
# -----------------------------
# 4. Seq2Seq Model
# -----------------------------
model = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 128)  │  1,280,000 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 100)       │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     49,408 │ embedding[0][0],  │
│                     │ (None, 64),       │            │ not_equal[0][0]   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 100, 64), │     16,896 │ decoder_input[0]… │
│                     │ (None, 64),       │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100, 1)    │         65 │ decoder_lstm[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,346,369 (5.14 MB)

 Trainable params: 1,346,369 (5.14 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# -----------------------------
# 5. Prepare Decoder Inputs (Teacher Forcing)
# -----------------------------
decoder_input_train = np.zeros((X_train.shape[0], max_len, 1))
decoder_input_test = np.zeros((X_test.shape[0], max_len, 1))

# Shifted target
decoder_target_train = y_train_seq[:, :, None]
decoder_target_test = y_test_seq[:, :, None]

In [10]:
X_train.shape

(25000, 100)

In [8]:
decoder_input_train.shape

(25000, 100, 1)

In [15]:
# -----------------------------
# 6. Train
# -----------------------------
model.fit(
    [X_train, decoder_input_train],
    decoder_target_train,
    batch_size=64,
    epochs=3,
    validation_data=([X_test, decoder_input_test], decoder_target_test)
)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.6945 - loss: 0.5659 - val_accuracy: 0.8364 - val_loss: 0.3912
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.8710 - loss: 0.3256 - val_accuracy: 0.8373 - val_loss: 0.3904
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.9148 - loss: 0.2425 - val_accuracy: 0.8440 - val_loss: 0.3638
